# Filter step4_MatchingONTDtoOSM.csv against B-o-T stop_times

`step4_MatchingONTDtoOSM.csv` is the output of an ONTD-to-OSM stop matching
pipeline (one row per ONTD stop, with its best-matching OSM stop). This
notebook keeps only the rows whose stop also appears in
`B-o-T_DataBase_stop_times.csv` — the actual stop_times export, where the
`stop_id` column holds a stop *name* (not a code) and the same stop repeats
once per trip visiting it, so it's deduplicated first.

A row is kept when, after deduplicating B-o-T's stops:
- its `ontd_name_ascii` matches a B-o-T stop name (accent/case/whitespace insensitive), **and**
- the ONTD coordinates (`ontd_lat`/`ontd_lon`) are within `MAX_DISTANCE_KM` of that B-o-T stop's coordinates (`stop_lat`/`stop_lon`)

The coordinate check guards against two unrelated stops sharing a name.

B-o-T's stop names are also repaired for a UTF-8-as-CP1252 mojibake
(`TimiÅŸoara Nord` -> `Timișoara Nord`) that shows up in the export, since
otherwise names with diacritics would silently fail to match.

In [8]:
import csv
import math
import unicodedata
from pathlib import Path

PROGRESS_EVERY = 2000

## Parameters

Adjust the paths and distance tolerance here, then run all cells below.

In [9]:
DATA_DIR = Path(r"C:\Users\glanz\Documents\BackOnTrack\night-train-target-network\backend\models\infrastructure\stop_classification\data")

STEP4_PATH = DATA_DIR / "step4_MatchingONTDtoOSM.csv"
STOP_TIMES_PATH = DATA_DIR / "B-o-T_DataBase_stop_times.csv"
OUTPUT_PATH = DATA_DIR / "step5_JoinedNTStops.csv"
UNMATCHED_OUTPUT_PATH = DATA_DIR / "unmatched_stops.csv"
STOP_TIMES_CHECK_OUTPUT_PATH = DATA_DIR / "stop_times_check.csv"
MAX_DISTANCE_KM = 50.0
COORD_FALLBACK_KM = 1.0  # used when the name doesn't match: fall back to pure coordinate proximity

## Helper functions

In [10]:
def fix_mojibake(text):
    """Repair UTF-8 text that was previously mis-decoded as CP1252.

    Round-tripping through cp1252 is only attempted when the text has
    non-ASCII characters, and is discarded unless it decodes cleanly as
    UTF-8 — genuine single-byte Western accents (e.g. plain "café") don't
    form valid multi-byte UTF-8 sequences and safely fail the round-trip,
    so this only ever changes text that was actually mis-decoded.
    """
    if not text or text.isascii():
        return text
    try:
        repaired = text.encode("cp1252").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return text
    return repaired if "\ufffd" not in repaired else text


def normalize_name(text):
    """Lowercase, strip accents/whitespace for tolerant name comparison."""
    if not text:
        return ""
    stripped = unicodedata.normalize("NFKD", text)
    ascii_only = stripped.encode("ascii", "ignore").decode("ascii")
    return " ".join(ascii_only.lower().split())


def parse_float(value):
    if value is None:
        return None
    value = value.strip()
    if not value:
        return None
    try:
        return float(value.replace(",", "."))
    except ValueError:
        return None


def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(
        dlambda / 2
    ) ** 2
    return 2 * r * math.asin(math.sqrt(a))

## Load and deduplicate B-o-T stops

In [11]:
def load_bot_stops(path):
    """Return {normalized_name: {"name": display_name, "coords": {(lat, lon), ...}}}.

    Deduplicated across the whole file — a stop repeated across many trips
    (or many times within one trip) collapses to a single entry.
    """
    stops_by_name = {}
    total_rows = 0
    with open(path, encoding="utf-8-sig", newline="") as f:
        for row in csv.DictReader(f):
            total_rows += 1
            if total_rows % PROGRESS_EVERY == 0:
                print(f"  B-o-T stop_times: {total_rows} rows processed...")
            name = fix_mojibake(row.get("stop_id", ""))
            norm = normalize_name(name)
            lat = parse_float(row.get("stop_lat"))
            lon = parse_float(row.get("stop_lon"))
            if not norm or lat is None or lon is None:
                continue
            entry = stops_by_name.setdefault(norm, {"name": name, "coords": set()})
            entry["coords"].add((lat, lon))
    unique_coords = sum(len(v["coords"]) for v in stops_by_name.values())
    print(
        f"B-o-T stop_times: {total_rows} rows -> "
        f"{len(stops_by_name)} unique stop names "
        f"({unique_coords} distinct coordinate pairs)"
    )
    return stops_by_name


def write_stop_times_check(path, stops_by_name):
    """Write every deduplicated B-o-T stop (name + coordinates) for inspection."""
    rows = []
    for entry in stops_by_name.values():
        for lat, lon in sorted(entry["coords"]):
            rows.append({"stop_name": entry["name"], "stop_lat": lat, "stop_lon": lon})
    rows.sort(key=lambda r: r["stop_name"])

    with open(path, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["stop_name", "stop_lat", "stop_lon"])
        writer.writeheader()
        writer.writerows(rows)

    print(f"B-o-T stop_times check: {len(rows)} rows -> {path}")
    return rows


stops_by_name = load_bot_stops(STOP_TIMES_PATH)
write_stop_times_check(STOP_TIMES_CHECK_OUTPUT_PATH, stops_by_name)

  B-o-T stop_times: 2000 rows processed...
B-o-T stop_times: 3869 rows -> 600 unique stop names (650 distinct coordinate pairs)
B-o-T stop_times check: 650 rows -> C:\Users\glanz\Documents\BackOnTrack\night-train-target-network\backend\models\infrastructure\stop_classification\data\stop_times_check.csv


[{'stop_name': 'Aachen Hbf', 'stop_lat': 50.76794, 'stop_lon': 6.09126},
 {'stop_name': 'Aachen Hbf', 'stop_lat': 50.76794, 'stop_lon': 6.09126317},
 {'stop_name': 'Aarau', 'stop_lat': 47.39043, 'stop_lon': 8.0457015},
 {'stop_name': 'Aberdeen', 'stop_lat': 57.14469, 'stop_lon': -2.096398882},
 {'stop_name': 'Afyon A Çetinkaya',
  'stop_lat': 38.76423,
  'stop_lon': 30.55250287},
 {'stop_name': 'Alba Iulia', 'stop_lat': 46.05756, 'stop_lon': 23.57650205},
 {'stop_name': 'Albi', 'stop_lat': 43.92487, 'stop_lon': 2.1486789},
 {'stop_name': 'Albi', 'stop_lat': 43.92487, 'stop_lon': 2.14868},
 {'stop_name': 'Albi Ville', 'stop_lat': 43.92445, 'stop_lon': 2.138167365},
 {'stop_name': 'Alvesta', 'stop_lat': 56.89914, 'stop_lon': 14.55715022},
 {'stop_name': 'Amersfoort Centraal',
  'stop_lat': 52.15356,
  'stop_lon': 5.374515941},
 {'stop_name': 'Amsterdam Centraal',
  'stop_lat': 52.37921,
  'stop_lon': 4.900292591},
 {'stop_name': 'Amstetten (NÖ)',
  'stop_lat': 48.12228,
  'stop_lon': 14.

## Filter step4_MatchingONTDtoOSM.csv

In [12]:
def dedupe_nearest_candidate(rows):
    """Keep only the smallest-distance_km row per ontd_id.

    step4 can list several OSM candidates per ONTD stop; only the nearest
    one (to the ONTD coordinate) is a valid match, never the others.
    """
    best_by_id = {}
    for row in rows:
        ontd_id = row.get("ontd_id")
        dist = parse_float(row.get("distance_km"))
        if dist is None:
            dist = float("inf")
        current = best_by_id.get(ontd_id)
        if current is None or dist < current[0]:
            best_by_id[ontd_id] = (dist, row)
    deduped = [row for _, row in best_by_id.values()]
    print(
        f"step4_MatchingONTDtoOSM: {len(rows)} candidate rows -> "
        f"{len(deduped)} rows (nearest OSM candidate per ontd_id)"
    )
    return deduped


def filter_step4(path, stops_by_name, max_distance_km, coord_fallback_km):
    """Return (fieldnames, kept_rows, matched_names, near_miss).

    Each B-o-T stop ends up matched to at most one step4 row: every step4
    row's best candidate B-o-T stop is computed first, then — if several
    step4 rows target the same B-o-T stop — only the closest one is kept
    and the rest are dropped, so B-o-T stops and kept step4 rows are in a
    strict 1:1 relationship (no "nearby stations also get picked").

    matched_names: normalized B-o-T stop names that got a kept row (via a
    name match or a coordinate-fallback match).
    near_miss: {normalized_name: closest_distance_km} for names that matched
    by name in step4 but whose closest candidate exceeded max_distance_km.
    """
    with open(path, encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames
        rows = list(reader)
    rows = dedupe_nearest_candidate(rows)

    near_miss = {}
    candidates = []  # (bot_norm, distance_km, row)
    for i, row in enumerate(rows, start=1):
        if i % PROGRESS_EVERY == 0:
            print(f"  step4_MatchingONTDtoOSM: {i}/{len(rows)} rows processed...")
        norm = normalize_name(row.get("ontd_name_ascii", ""))
        lat = parse_float(row.get("ontd_lat"))
        lon = parse_float(row.get("ontd_lon"))
        if lat is None or lon is None:
            continue

        best_norm, best_dist = None, None
        entry = stops_by_name.get(norm)
        if entry:
            closest = min(haversine_km(lat, lon, c_lat, c_lon) for c_lat, c_lon in entry["coords"])
            if closest <= max_distance_km:
                best_norm, best_dist = norm, closest
            elif closest < near_miss.get(norm, float("inf")):
                near_miss[norm] = closest

        if best_norm is None:
            # Coordinate fallback: try every B-o-T stop's coordinates,
            # regardless of name.
            fb_norm, fb_dist = None, coord_fallback_km
            for norm2, entry2 in stops_by_name.items():
                for c_lat, c_lon in entry2["coords"]:
                    d = haversine_km(lat, lon, c_lat, c_lon)
                    if d <= fb_dist:
                        fb_dist = d
                        fb_norm = norm2
            best_norm, best_dist = fb_norm, fb_dist

        if best_norm is not None:
            candidates.append((best_norm, best_dist, row))

    # Enforce the 1:1 relationship: per B-o-T stop, keep only the closest
    # candidate row and drop every other step4 row aimed at it.
    best_per_bot_stop = {}
    for norm, dist, row in candidates:
        current = best_per_bot_stop.get(norm)
        if current is None or dist < current[0]:
            best_per_bot_stop[norm] = (dist, row)

    kept = [row for _, row in best_per_bot_stop.values()]
    matched_names = set(best_per_bot_stop.keys())

    print(f"step4_MatchingONTDtoOSM: {len(rows)} rows -> {len(kept)} kept")
    return fieldnames, kept, matched_names, near_miss


fieldnames, kept_rows, matched_names, near_miss = filter_step4(
    STEP4_PATH, stops_by_name, MAX_DISTANCE_KM, COORD_FALLBACK_KM
)

step4_MatchingONTDtoOSM: 48617 candidate rows -> 48617 rows (nearest OSM candidate per ontd_id)
  step4_MatchingONTDtoOSM: 2000/48617 rows processed...
  step4_MatchingONTDtoOSM: 4000/48617 rows processed...
  step4_MatchingONTDtoOSM: 6000/48617 rows processed...
  step4_MatchingONTDtoOSM: 8000/48617 rows processed...
  step4_MatchingONTDtoOSM: 10000/48617 rows processed...
  step4_MatchingONTDtoOSM: 12000/48617 rows processed...
  step4_MatchingONTDtoOSM: 14000/48617 rows processed...
  step4_MatchingONTDtoOSM: 16000/48617 rows processed...
  step4_MatchingONTDtoOSM: 18000/48617 rows processed...
  step4_MatchingONTDtoOSM: 20000/48617 rows processed...
  step4_MatchingONTDtoOSM: 22000/48617 rows processed...
  step4_MatchingONTDtoOSM: 24000/48617 rows processed...
  step4_MatchingONTDtoOSM: 26000/48617 rows processed...
  step4_MatchingONTDtoOSM: 28000/48617 rows processed...
  step4_MatchingONTDtoOSM: 30000/48617 rows processed...
  step4_MatchingONTDtoOSM: 32000/48617 rows processed

## Write the filtered CSV

In [13]:
with open(OUTPUT_PATH, "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(kept_rows)

print(f"Wrote {len(kept_rows)} rows to {OUTPUT_PATH}")

Wrote 568 rows to C:\Users\glanz\Documents\BackOnTrack\night-train-target-network\backend\models\infrastructure\stop_classification\data\step5_JoinedNTStops.csv


## B-o-T stops with no match

Every B-o-T stop that ended up with no accepted match in step4, split into:
- `no_name_match_in_step4` — the name doesn't appear in step4 at all
- `name_matched_but_too_far` — the name appears, but every candidate was farther than `MAX_DISTANCE_KM` (closest distance shown, useful for tuning the threshold or spotting bad coordinates)

In [14]:
def write_unmatched_report(path, stops_by_name, matched_names, near_miss):
    """Write every B-o-T stop with no accepted match, with a reason."""
    rows = []
    for norm, entry in stops_by_name.items():
        if norm in matched_names:
            continue
        closest = near_miss.get(norm)
        reason = "name_matched_but_too_far" if closest is not None else "no_name_match_in_step4"
        for lat, lon in sorted(entry["coords"]):
            rows.append(
                {
                    "stop_name": entry["name"],
                    "stop_lat": lat,
                    "stop_lon": lon,
                    "reason": reason,
                    "closest_distance_km": f"{closest:.2f}" if closest is not None else "",
                }
            )
    rows.sort(key=lambda r: r["stop_name"])

    with open(path, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["stop_name", "stop_lat", "stop_lon", "reason", "closest_distance_km"],
        )
        writer.writeheader()
        writer.writerows(rows)

    print(f"B-o-T stops with no match: {len(rows)} rows -> {path}")
    return rows


unmatched_rows = write_unmatched_report(UNMATCHED_OUTPUT_PATH, stops_by_name, matched_names, near_miss)
unmatched_rows[:20]

B-o-T stops with no match: 34 rows -> C:\Users\glanz\Documents\BackOnTrack\night-train-target-network\backend\models\infrastructure\stop_classification\data\unmatched_stops.csv


[{'stop_name': 'Albi',
  'stop_lat': 43.92487,
  'stop_lon': 2.1486789,
  'reason': 'no_name_match_in_step4',
  'closest_distance_km': ''},
 {'stop_name': 'Albi',
  'stop_lat': 43.92487,
  'stop_lon': 2.14868,
  'reason': 'no_name_match_in_step4',
  'closest_distance_km': ''},
 {'stop_name': 'Amsterdam Centraal',
  'stop_lat': 52.37921,
  'stop_lon': 4.900292591,
  'reason': 'no_name_match_in_step4',
  'closest_distance_km': ''},
 {'stop_name': 'Beograd',
  'stop_lat': 44.79361,
  'stop_lon': 20.4533295,
  'reason': 'no_name_match_in_step4',
  'closest_distance_km': ''},
 {'stop_name': 'Briançon',
  'stop_lat': 44.88961,
  'stop_lon': 6.632627,
  'reason': 'no_name_match_in_step4',
  'closest_distance_km': ''},
 {'stop_name': 'Burgas',
  'stop_lat': 45.15,
  'stop_lon': 27.472671,
  'reason': 'no_name_match_in_step4',
  'closest_distance_km': ''},
 {'stop_name': 'Chervonohrad',
  'stop_lat': 50.38743,
  'stop_lon': 24.22281595,
  'reason': 'no_name_match_in_step4',
  'closest_distance_